In [ ]:
using DelimitedFiles
using Statistics
using Printf

const OUTDIR   = "output"
const NCHAINS  = 3
const BURN_IN_LOGLIK = 0  
const THIN_LOGLIK    = 1

# ---------- I/O ----------
function read_csv_matrix(path::String)
    data, hdr = DelimitedFiles.readdlm(path, ',', header=true)
    M = Matrix{Float64}(data)
    header = hdr === nothing ? String[] : String.(vec(hdr))
    return M, header
end

function read_best3(path::String)::Dict{Int,Vector{Int}}
    data, hdr = DelimitedFiles.readdlm(path, ',', header=true)
    H = hdr === nothing ? String[] : String.(vec(hdr))
    colidx = Dict(name => findfirst(==(name), H) for name in H)

    req = ["Dataset","kmin1","kmin2","kmin3"]
    for r in req
        haskey(colidx, r) || error("Column $r not found in $path")
    end

    to_int_or_nothing(x) = (x isa Number && !isnan(x)) ? Int(round(x)) : nothing

    ks_by_sim = Dict{Int, Vector{Int}}()
    for r in 1:size(data,1)
        sim_val = data[r, colidx["Dataset"]]
        sim = to_int_or_nothing(sim_val)
        sim === nothing && error("Invalid Dataset value at row $r: $sim_val")

        kvec = Int[]
        for col in ("kmin1","kmin2","kmin3")
            v = data[r, colidx[col]]
            kin = to_int_or_nothing(v)
            if kin !== nothing
                push!(kvec, kin)
            end
        end
        ks_by_sim[sim] = kvec
    end
    return ks_by_sim
end

function load_loglik(sim::Int, k::Int, chain::Int)::Matrix{Float64}
    path = joinpath(OUTDIR, @sprintf("loglik_sim_%d_chain_%d_k%d.csv", sim, chain, k))
    if !isfile(path)
        @warn "Missing loglik: $path"
        return Array{Float64}(undef, 0, 0)
    end
    M, _ = read_csv_matrix(path)
    if BURN_IN_LOGLIK > 0 || THIN_LOGLIK > 1
        idxs = collect(BURN_IN_LOGLIK+1:THIN_LOGLIK:size(M,1))
        M = M[idxs, :]
    end
    return M
end

# ---------- WAIC ----------
function logmeanexp_columnwise(L::AbstractMatrix{<:Real})
    S, T = size(L)
    out = Vector{Float64}(undef, T)
    @inbounds for j in 1:T
        col = @view L[:, j]
        m = maximum(col)
        out[j] = m + log(sum(exp.(col .- m)) / S)
    end
    return out
end

function compute_waic(L::AbstractMatrix{<:Real})
    isempty(L) && return NaN
    if any(isinf, L)
        @warn "Inf log-likelihood encountered; WAIC -> -Inf"
        return -Inf
    end
    lppd  = sum(logmeanexp_columnwise(L))
    pwaic = sum(var(L, dims=1))
    return -2*lppd + 2*pwaic
end


best3_path = "best3_by_dataset.csv"  
ks_by_sim  = read_best3(best3_path)

rows = String[]
push!(rows, "Dataset,k1,WAIC1,k2,WAIC2,k3,WAIC3,best_k")

for sim in sort(collect(keys(ks_by_sim)))
    ks_raw = ks_by_sim[sim]


    seen = Set{Int}()
    ks = Int[]
    for k in ks_raw
        if !(k in seen)
            push!(seen, k); push!(ks, k)
        end
        if length(ks) == 3
            break
        end
    end

    waics = Float64[]
    for k in ks
        println(@sprintf("Processing sim=%d, k=%d ...", sim, k))

        parts = Matrix{Float64}[]
        for c in 1:NCHAINS
            M = load_loglik(sim, k, c)
            if size(M,1) > 0
                push!(parts, M)
            else
                @warn "Empty loglik for sim=$sim, k=$k, chain=$c"
            end
        end
        L = isempty(parts) ? Array{Float64}(undef,0,0) : vcat(parts...)
        push!(waics, compute_waic(L))
    end


    while length(ks) < 3
        push!(ks, Int(typemax(Int)))  
        push!(waics, NaN)
    end

    best_k = if all(isnan, waics)
        NaN
    else
        ks[argmin(waics)]
    end

    line = @sprintf("%d,%d,%.12g,%d,%.12g,%d,%.12g,%s",
                    sim, ks[1], waics[1], ks[2], waics[2], ks[3], waics[3],
                    string(best_k))
    push!(rows, line)
end

outcsv = joinpath(OUTDIR, "waic_best3_mergedchains.csv")
open(outcsv, "w") do io
    for r in rows
        println(io, r)
    end
end

@info "Done -> $(outcsv)"


Processing sim=1, k=13 ...
Processing sim=1, k=14 ...
Processing sim=1, k=15 ...
Processing sim=2, k=14 ...
Processing sim=2, k=15 ...
Processing sim=2, k=13 ...
Processing sim=3, k=14 ...
Processing sim=3, k=15 ...
Processing sim=3, k=13 ...
Processing sim=4, k=13 ...
Processing sim=4, k=14 ...
Processing sim=4, k=15 ...
Processing sim=5, k=14 ...
Processing sim=5, k=13 ...
Processing sim=5, k=15 ...
Processing sim=6, k=14 ...
Processing sim=6, k=13 ...
Processing sim=6, k=12 ...
Processing sim=7, k=13 ...
Processing sim=7, k=15 ...
Processing sim=7, k=14 ...
Processing sim=8, k=13 ...
Processing sim=8, k=14 ...
Processing sim=8, k=12 ...
Processing sim=9, k=13 ...
Processing sim=9, k=14 ...
Processing sim=9, k=15 ...
Processing sim=10, k=14 ...
Processing sim=10, k=15 ...
Processing sim=10, k=13 ...
Processing sim=11, k=14 ...
Processing sim=11, k=13 ...
Processing sim=11, k=15 ...
Processing sim=12, k=14 ...
Processing sim=12, k=13 ...
Processing sim=12, k=15 ...
Processing sim=13, 

[ Info: Done -> output/waic_best3_mergedchains.csv


In [ ]:
using CSV, DataFrames, Statistics, Printf, StatsBase
include("functions.jl")   

# ---------------- Config ----------------
data_dir  = "output"
burnin    = 500_000
thin      = 10
chain_ids = 1:3
outfile   = joinpath(data_dir, "rhat_all.csv")
# ----------------------------------------

function load_chain_df(path::String; burnin::Int, thin::Int)
    df = CSV.read(path, DataFrame)
    if nrow(df) <= burnin
        @warn "File has only $(nrow(df)) rows (<= burn-in): $path"
        return DataFrame() 
    end
    idx = (burnin + 1):thin:nrow(df)
    return df[idx, :]
end

function index_samples(data_dir::String)
    rx = r"^samples_sim_(\d+)_chain_(\d+)_k(\d+)\.csv$"
    idx = Dict{Int, Dict{Int, Dict{Int,String}}}()
    for fname in readdir(data_dir)
        m = match(rx, fname)
        m === nothing && continue
        sim   = parse(Int, m.captures[1])
        chain = parse(Int, m.captures[2])
        k     = parse(Int, m.captures[3])
        get!(idx, sim, Dict{Int, Dict{Int,String}}())
        get!(idx[sim], k, Dict{Int,String}())
        idx[sim][k][chain] = joinpath(data_dir, fname)
    end
    return idx
end

function rhat_for_sim_k(sim::Int, k::Int, chain_paths::Dict{Int,String};
                        burnin::Int, thin::Int, chain_ids=1:3)
    dfs = DataFrame[]
    used = Int[]
    for c in chain_ids
        haskey(chain_paths, c) || continue
        df = load_chain_df(chain_paths[c]; burnin=burnin, thin=thin)
        if nrow(df) == 0
            @warn "No usable rows after burn/thin: sim=$sim k=$k chain=$c"
            continue
        end
        push!(dfs, df)
        push!(used, c)
    end
    if length(dfs) < 2
        @warn "sim=$sim k=$k has <2 usable chains; skipping."
        return nothing
    end

    common_params = reduce(intersect, map(names, dfs))
    if isempty(common_params)
        @warn "sim=$sim k=$k: no common parameters across chains; skipping."
        return nothing
    end

    n_keep = minimum(nrow.(dfs))
    dfs = [df[1:n_keep, common_params] for df in dfs]

    gr_dict = Dict{String,Float64}()
    for p in String.(common_params)
        mat = Array{Float64}(undef, length(dfs), n_keep)
        for (i, df) in enumerate(dfs)
            mat[i, :] = Float64.(df[!, p])
        end
        gr_dict[p] = rhat_gelman_rubin(mat)
    end
    return gr_dict
end

idx = index_samples(data_dir)
if isempty(idx)
    @warn "No matching sample files found under: $data_dir"
end

open(outfile, "w") do io
    println(io, "Dataset,k,Parameter,Rhat")
    for sim in sort(collect(keys(idx)))
        ks = sort(collect(keys(idx[sim])))
        for k in ks
            @info(@sprintf("Computing R̂ for dataset %d, k=%d", sim, k))
            gr = rhat_for_sim_k(sim, k, idx[sim][k]; burnin=burnin, thin=thin, chain_ids=chain_ids)
            gr === nothing && continue
            for (param, rhat) in gr
                @printf(io, "%d,%d,%s,%.6f\n", sim, k, param, rhat)
            end
        end
    end
end

@info "R̂ written to $(outfile)"


[ Info: Computing R̂ for dataset 1, k=13
[ Info: Computing R̂ for dataset 1, k=14
[ Info: Computing R̂ for dataset 1, k=15
[ Info: Computing R̂ for dataset 2, k=13
[ Info: Computing R̂ for dataset 2, k=14
[ Info: Computing R̂ for dataset 2, k=15
[ Info: Computing R̂ for dataset 3, k=13
[ Info: Computing R̂ for dataset 3, k=14
[ Info: Computing R̂ for dataset 3, k=15
[ Info: Computing R̂ for dataset 4, k=13
[ Info: Computing R̂ for dataset 4, k=14
[ Info: Computing R̂ for dataset 4, k=15
[ Info: Computing R̂ for dataset 5, k=13
[ Info: Computing R̂ for dataset 5, k=14
[ Info: Computing R̂ for dataset 5, k=15
[ Info: Computing R̂ for dataset 6, k=12
[ Info: Computing R̂ for dataset 6, k=13
[ Info: Computing R̂ for dataset 6, k=14
[ Info: Computing R̂ for dataset 7, k=13
[ Info: Computing R̂ for dataset 7, k=14
[ Info: Computing R̂ for dataset 7, k=15
[ Info: Computing R̂ for dataset 8, k=12
[ Info: Computing R̂ for dataset 8, k=13
[ Info: Computing R̂ for dataset 8, k=14
[ Info: Computin